# MSH chain on a 2D substrate: tensor networks

**Goal.** Measure how the cost of the tensor-network (MPO/MPS) methods grows with system size, against the dense and sparse
Python methods of `msh_substrate_python.ipynb`, for an MSH chain on a 2D superconducting substrate. The substrate is grown first in
width (more substrate around the same chain), then as a whole.

**Model.** A $2^{L_x}\times2^{L_y}$ square substrate with hopping $t = 1$, chemical potential $\mu = -7$, Rashba $\alpha = 0.9$ and
s-wave pairing $\Delta = 2.4$ on every site. The magnetic chain is the centred segment of $\ell$ sites in the middle row
$i_y = N_y/2$, with exchange $J\,\mathbf S\cdot\boldsymbol\sigma$, $J = 5.8$. With $L_y = 0$ (one row) this is exactly the MSH
chain of `msh_chain_tn.ipynb` with a centred topological segment. At $\mu = -7$ the bare substrate band is empty, so the substrate
is a gapped, trivial superconductor; the chain stays topological on it (`msh_substrate_python.ipynb`, section 3). At this point the
zero modes barely enter the substrate (gap $\approx 0.55$, decay within one row), which favours the tensor network. Section 8 repeats
the benchmark at a second point, $\Delta = 0.6$, $\mu = -3$ (inside the band), $J = 3$, where they spread over several rows.

**Contents**
1. Setup and helpers
2. Geometry
3. Hamiltonian MPO
4. Check data for the Python notebook
5. MPO size and build time against system size
6. Zero-mode pipeline
7. Zero-mode benchmark
8. A substrate the zero modes leak into ($\Delta = 0.6$, $\mu = -3$, $J = 3$)

**Conventions.** Sites `[nambu; spin; y₁ … y_Ly; x₁ … x_Lx]`, most significant bit first, so the position index is
$n = i_x + i_y N_x$ (TensorBinding's 2D row-major convention). Nambu basis $(c_\uparrow, c_\downarrow, c^\dagger_\uparrow, c^\dagger_\downarrow)$,
as in the other Julia notebooks; the Python notebook uses kwant's $(c_\uparrow, c_\downarrow, c^\dagger_\downarrow, -c^\dagger_\uparrow)$.

## 1. Setup and helpers

In [1]:
using TensorBinding, ITensors, ITensorMPS, LinearAlgebra, Printf, NPZ, Random, Logging, DelimitedFiles
const TB = TensorBinding
BLAS.set_num_threads(1)            # single thread, as in the Python notebook, so that timings compare like with like

const σ0 = ComplexF64[1 0; 0 1]
const σx = ComplexF64[0 1; 1 0]
const σy = ComplexF64[0 -im; im 0]
const σz = ComplexF64[1 0; 0 -1]

const DATA_DIR = "data"            # check data for msh_substrate_python.ipynb (not committed)
mkpath(DATA_DIR)

# diagonal MPO diag(ψ) of an MPS ψ on the position qubits: each site tensor is copied onto the diagonal
function diag_mpo(ψ::MPS, pos)
    W = MPO(pos, "Id")
    for k in eachindex(pos)
        s = pos[k]
        δ = ITensor(ComplexF64, s, s', dag(s''))
        for j in 1:2; δ[s => j, s' => j, s'' => j] = 1; end
        W[k] = replaceprime(ψ[k] * δ, 2 => 0; tags=tags(s))
    end
    return W
end

# MPS of a vector v on qubits `q` (q[1] = most significant bit), exact up to `cutoff`; builds the 2^length(q) array
vector_mps(v::AbstractVector, q; cutoff=1e-14) =
    ITensors.@disable_warn_order MPS(permutedims(reshape(ComplexF64.(v), ntuple(_ -> 2, length(q))), length(q):-1:1), q; cutoff)

# exact diagonal MPO diag(v) on the position qubits (pos[1] = most significant bit); O(length(v)) work
diag_mpo_exact(v::AbstractVector, pos; cutoff=1e-14) = diag_mpo(vector_mps(v, pos; cutoff), pos)

# row_profile_mps(yvecs, w, ypos, xpos): MPS of (yvecs[1] ⊗ … ⊗ yvecs[Ly]) ⊗ w on the qubits [ypos; xpos], i.e. a product of
# single-qubit vectors on the y qubits times a profile w (length 2^Lx) on the x qubits. Costs O(2^Lx), not O(N): only w is
# compressed. With yvecs = the bits of one row it is nonzero on that row; with yvecs = (1, 1) it is the same profile on every row.
function row_profile_mps(yvecs, w, ypos, xpos; cutoff=1e-14)
    ψx = vector_mps(w, xpos; cutoff)
    isempty(ypos) && return ψx
    link = [Index(1, "Link,ly=$k") for k in eachindex(ypos)]          # dimension-1 bonds through the product part
    T = ITensor[]
    for (k, s) in enumerate(ypos)
        t = ITensor(ComplexF64.(yvecs[k]), s) * onehot(link[k] => 1)
        k > 1 && (t *= onehot(link[k-1] => 1))
        push!(T, t)
    end
    ψx[1] *= onehot(link[end] => 1)
    return MPS(vcat(T, collect(ψx)))
end

# dense matrix of an MPO (big-endian over `sites`); small systems only
dense(mpo, sites) = (T = ITensors.@disable_warn_order prod(mpo); D = prod(dim.(sites));
                     reshape(Array(T, reverse(prime.(sites))..., reverse(sites)...), D, D))
adjoint_mpo(K) = swapprime(dag(K), 0, 1)

adjoint_mpo (generic function with 1 method)

## 2. Geometry

`Substrate(Lx, Ly, ℓ)`: $N_x = 2^{L_x}$ columns, $N_y = 2^{L_y}$ rows, and the chain of $\ell$ sites centred in row $N_y/2$.
`chain_x` lists the (0-indexed) columns of the chain sites.

In [2]:
struct Substrate
    Lx::Int; Ly::Int; ℓ::Int
end
Nx(g::Substrate) = 2^g.Lx
Ny(g::Substrate) = 2^g.Ly
nsites(g::Substrate) = Nx(g) * Ny(g)
chain_row(g::Substrate) = Ny(g) ÷ 2
chain_x(g::Substrate) = ((Nx(g) - g.ℓ) ÷ 2):((Nx(g) - g.ℓ) ÷ 2 + g.ℓ - 1)

# spin textures of the chain: all along ẑ (topological), and a generic tilted one for the Hamiltonian check
S_z(ix) = (0.0, 0.0, 1.0)
S_tilt(ix) = (sin(0.3ix) * cos(0.7), sin(0.3ix) * sin(0.7), cos(0.3ix))

S_tilt (generic function with 1 method)

## 3. Hamiltonian MPO

Particle block
$$H_0 = \sum_r c_r^\dagger(-\mu)c_r + \sum_r\bigl[c_{r+\hat x}^\dagger h_x c_r + c_{r+\hat y}^\dagger h_y c_r + \text{h.c.}\bigr]
+ J\sum_{r\in\text{chain}} c_r^\dagger\,\mathbf S(r)\cdot\boldsymbol\sigma\,c_r,\qquad
h_x = -t + i\alpha\sigma_y,\quad h_y = -t - i\alpha\sigma_x,$$
hole block $-H_0^*$ and singlet pairing $\Delta\,i\sigma_y$, assembled as in `msh_chain` of `msh_chain_tn.ipynb`.

- **x bonds:** the open binary shift $|n\rangle\to|n+1\rangle$ on all position qubits, multiplied by a mask that removes the last
  column, so that $i_x = N_x - 1$ does not hop to $i_x = 0$ of the next row. The mask is the same on every row:
  $(1,1)^{\otimes L_y}\otimes m(i_x)$ with $m = 0$ at $i_x = N_x-1$ and 1 elsewhere.
- **y bonds:** the open shift $|n\rangle\to|n+N_x\rangle$, which maps the top row to zero (open boundary). With one row
  ($L_y = 0$) there are no y bonds; `shift_pair_mpos(pos, Nx)` would then keep only the low bits of $N_x = 2^{L}$, i.e. act as the
  identity, so it must not be called.
- **Exchange:** one diagonal MPO per spin component, $e_{i_{y0}}\otimes JS_k(i_x)$: nonzero on the chain row only, with the chain profile
  on the x qubits.

Both diagonals are built with `row_profile_mps` (a product state on the y qubits times a compressed profile of length $N_x$ on the x
qubits) and `diag_mpo`, so building $H$ costs $O(N_x)$, not $O(N)$. (An earlier version compressed full length-$N$ vectors with
`diag_mpo_exact`; at $512\times512$ and $1024\times1024$ sites that took 12–14 s of the build.)

In [3]:
"""
    msh_substrate(g, S; t, μ, α, Δ, J, cutoff, sites) -> (H_BdG::MPO, sites)

BdG MPO of the MSH chain (spin texture `S(ix)`) on the substrate `g`. `sites` reuses existing site indices.
"""
function msh_substrate(g::Substrate, S; t=1.0, μ=-7.0, α=0.9, Δ=2.4, J=5.8, cutoff=1e-13, sites=nothing)
    nx = Nx(g)
    ns, ss, pos = isnothing(sites) ? (TB.nambu_index(), TB.spin_index(), siteinds("Qubit", g.Lx + g.Ly)) :
                                     (sites[1], sites[2], sites[3:end])
    ypos, xpos = pos[1:g.Ly], pos[g.Ly+1:end]
    Id = MPO(pos, "Id")
    P(op, M) = TB.prepend_spin(M, ss, op)
    every_row = [[1.0, 1.0] for _ in 1:g.Ly]
    row(iy) = [(b = (iy >> (g.Ly - k)) & 1; b == 0 ? [1.0, 0.0] : [0.0, 1.0]) for k in 1:g.Ly]

    K1 = first(TB.shift_pair_mpos(pos, 1; cyclic=false))
    mask = diag_mpo(row_profile_mps(every_row, [ix != nx - 1 ? 1.0 : 0.0 for ix in 0:nx-1], ypos, xpos), pos)
    Kx = apply(K1, mask; cutoff)
    bonds = [(Kx, -t * σ0 + im * α * σy)]
    g.Ly > 0 && push!(bonds, (first(TB.shift_pair_mpos(pos, nx; cyclic=false)), -t * σ0 - im * α * σx))

    terms = MPO[P(σ0, -μ * Id)]
    for (K, hop) in bonds                                         # c†_{r+d} hop c_r and its h.c.
        push!(terms, P(hop, K), P(Matrix(hop'), adjoint_mpo(K)))
    end
    xs, iy0 = chain_x(g), chain_row(g)
    for (k, σ) in enumerate((σx, σy, σz))
        w = zeros(nx)                                             # J S_k(ix) along the chain row
        for ix in xs
            w[ix + 1] = J * S(ix)[k]
        end
        maximum(abs, w) > 1e-14 && push!(terms, P(σ, diag_mpo(row_profile_mps(row(iy0), w, ypos, xpos), pos)))
    end
    H0 = +(terms...; cutoff)

    Hp  = P(:iSy, Δ * Id)
    HB  = +(TB.prepend_nambu(H0, ns, :Pp), TB.prepend_nambu(-1.0 * dag(H0), ns, :Ph),
            TB.prepend_nambu(Hp, ns, :tp), TB.prepend_nambu(adjoint_mpo(Hp), ns, :tm); cutoff)
    return HB, [ns; ss; pos]
end

msh_substrate (generic function with 1 method)

## 4. Check data for the Python notebook

Dense matrices of the MPO for ten small systems (one row; two to eight rows; a substrate taller than wide; the $\hat z$ and the tilted
texture), written to `data/tn_check_H.npz`. `msh_substrate_python.ipynb` (section 2) compares them element by element with its sparse
Hamiltonian after the change of Nambu basis.

In [4]:
check_cases = [(3, 0, 6), (3, 1, 6), (3, 2, 5), (2, 3, 3), (4, 2, 10)]
check = Dict{String,Matrix{ComplexF64}}()
for (Lx, Ly, ℓ) in check_cases, (name, S) in (("z", S_z), ("tilt", S_tilt))
    HB, sites = msh_substrate(Substrate(Lx, Ly, ℓ), S)
    H = dense(HB, sites)
    check["H_$(Lx)_$(Ly)_$(ℓ)_$name"] = H
    @printf("Lx = %d, Ly = %d, ℓ = %2d, %-4s: D = %3d, χ(H) = %d, ‖H − H†‖ = %.1e\n", Lx, Ly, ℓ, name, size(H, 1), maxlinkdim(HB), norm(H - H'))
end
npzwrite(joinpath(DATA_DIR, "tn_check_H.npz"), check)
println("wrote ", joinpath(DATA_DIR, "tn_check_H.npz"))

Lx = 3, Ly = 0, ℓ =  6, z   : D =  32, χ(H) = 7, ‖H − H†‖ = 1.4e-13
Lx = 3, Ly = 0, ℓ =  6, tilt: D =  32, χ(H) = 8, ‖H − H†‖ = 3.0e-13
Lx = 3, Ly = 1, ℓ =  6, z   : D =  64, χ(H) = 7, ‖H − H†‖ = 3.5e-13
Lx = 3, Ly = 1, ℓ =  6, tilt: D =  64, χ(H) = 8, ‖H − H†‖ = 2.3e-13
Lx = 3, Ly = 2, ℓ =  5, z   : D = 128, χ(H) = 7, ‖H − H†‖ = 4.4e-13
Lx = 3, Ly = 2, ℓ =  5, tilt: D = 128, χ(H) = 8, ‖H − H†‖ = 5.2e-13
Lx = 2, Ly = 3, ℓ =  3, z   : D = 128, χ(H) = 7, ‖H − H†‖ = 3.7e-13
Lx = 2, Ly = 3, ℓ =  3, tilt: D = 128, χ(H) = 8, ‖H − H†‖ = 3.4e-13
Lx = 4, Ly = 2, ℓ = 10, z   : D = 256, χ(H) = 7, ‖H − H†‖ = 7.4e-13
Lx = 4, Ly = 2, ℓ = 10, tilt: D = 256, χ(H) = 9, ‖H − H†‖ = 9.0e-13
wrote data/tn_check_H.npz


## 5. MPO size and build time against system size

Width scan ($L_x = 6$, $L_y = 0\dots6$, $\ell = 48$) and square scan ($L_x = L_y = 4\dots11$, $\ell = 3N_x/4$, up to
$2048\times2048$ sites). The bond dimension of the Hamiltonian, $\chi(H)$, and its memory (number of stored elements of the MPO tensors)
are compared with the number of nonzeros of the sparse matrix (`msh_substrate_python.ipynb`, section 4). Build times are measured after
a warm-up build of each size, so they do not include compilation; with the $O(N_x)$ construction of section 3 they stay in the
millisecond range at every size.

In [5]:
mpo_size(H) = sum(length(storage(T)) for T in H)              # stored elements of the MPO tensors
scan = vcat([(6, Ly, 48) for Ly in 0:6], [(L, L, 3 * 2^L ÷ 4) for L in 4:11])
build = []
for (Lx, Ly, ℓ) in scan
    g = Substrate(Lx, Ly, ℓ)
    msh_substrate(g, S_z)                                        # warm-up
    t_build = @elapsed (HB = first(msh_substrate(g, S_z)))
    push!(build, (; Lx, Ly, ℓ, N = nsites(g), χ = maxlinkdim(HB), mem = mpo_size(HB), t_build))
    @printf("Lx = %d, Ly = %d: N = %6d sites, D = %7d, χ(H) = %d, MPO elements = %6d, build %.2f s\n",
            Lx, Ly, nsites(g), 4nsites(g), maxlinkdim(HB), mpo_size(HB), t_build)
end

Lx = 6, Ly = 0: N =     64 sites, D =     256, χ(H) = 7, MPO elements =    612, build 0.01 s
Lx = 6, Ly = 1: N =    128 sites, D =     512, χ(H) = 7, MPO elements =    724, build 0.01 s
Lx = 6, Ly = 2: N =    256 sites, D =    1024, χ(H) = 7, MPO elements =    860, build 0.01 s
Lx = 6, Ly = 3: N =    512 sites, D =    2048, χ(H) = 7, MPO elements =   1048, build 0.01 s
Lx = 6, Ly = 4: N =   1024 sites, D =    4096, χ(H) = 7, MPO elements =   1244, build 0.01 s
Lx = 6, Ly = 5: N =   2048 sites, D =    8192, χ(H) = 7, MPO elements =   1440, build 0.02 s
Lx = 6, Ly = 6: N =   4096 sites, D =   16384, χ(H) = 7, MPO elements =   1636, build 0.02 s
Lx = 4, Ly = 4: N =    256 sites, D =    1024, χ(H) = 7, MPO elements =   1036, build 0.01 s
Lx = 5, Ly = 5: N =   1024 sites, D =    4096, χ(H) = 7, MPO elements =   1340, build 0.01 s
Lx = 6, Ly = 6: N =   4096 sites, D =   16384, χ(H) = 7, MPO elements =   1636, build 0.02 s
Lx = 7, Ly = 7: N =  16384 sites, D =   65536, χ(H) = 7, MPO elements 

## 6. Zero-mode pipeline

The pipeline of `msh_chain_tn.ipynb` (sections 8–9), adapted to the substrate:

1. `zero_mode_subspace`: penalized DMRG on $K = H^2 + \eta^2$ ($\eta = 10^{-2}$) for the two near-zero states and a third state as a gap
   check; $E_{\rm gap} = \lVert H\psi_3\rVert$ should sit at the bulk gap. The time of building $K$ and of the three DMRG runs is
   returned separately. `cutoff_K = 1e-20`: see below. **Start of the second state** (below).
2. `fermion_from_subspace`: localize with the position operator $\hat X = \sum_k 2^{L_x-k}\hat n_k$ on the **x qubits only** (in 1D
   all position qubits are x qubits), fix $C\gamma = \gamma$, pair into $\psi_f = (\gamma_1 + i\gamma_2)/\sqrt2$ with $E_f > 0$.
3. $\Delta Q = \langle\psi_f|\tau_z|\psi_f\rangle$.

The benchmark uses $\ell = 12$: there $E_f \approx 4\times10^{-4}$ and $|\Delta Q| \approx 3\times10^{-3}$ on the substrate, both far above
round-off and the DMRG accuracy, so they measure accuracy. (For $\ell = 48$ the splitting is $\sim10^{-14}$ and $\Delta Q$ would be pure
round-off.) With $E_f \gg 2\times10^{-6}$ no inverse-iteration refinement is needed (`msh_chain_tn.ipynb`, section 9).

**Truncation of $K$.** $\Delta Q$ is first order in any admixture of bulk states in the pair (energies only second order), so it is the
quantity most sensitive to how $K$ is built. With `cutoff_K = 1e-14` the one-row chain ($L_y = 0$) came out with
$|\Delta Q_{\rm TN} - \Delta Q_{\rm exact}| = 1.3\times10^{-5}$: the pair states carried a weight $4.7\times10^{-10}$ outside the exact pair,
independent of the number of sweeps and of the DMRG cutoff, and gone with `cutoff_K = 1e-20` ($1.3\times10^{-10}$). With a substrate the
error was a few $\times10^{-9}$ for either value, limited by DMRG convergence. The benchmark therefore uses `cutoff_K = 1e-20` (as the
$\Delta Q$ calculations of `msh_chain_tn.ipynb` do); it doubles $\chi(K)$ (18 → 38) and makes the DMRG 1.2–1.6 times slower on the
large substrates, which is part of the cost of the method at this accuracy. The inverse-iteration step of `msh_chain_tn.ipynb` is not
an alternative here: with $E_f$ close to $\sigma = 10^{-3}$ it made $L_y = 1$ worse ($5.7\times10^{-5}$).

**Start of the second state.** With a random start, the DMRG for the second zero mode can stop in a wrong state: at the second
parameter point of section 8 and $2048\times2048$ sites it ended at $\lVert H\psi_2\rVert = 0.60$ (a substrate band-edge state, bond dimension
at the cap of 200), and the gap check picked up the missing zero mode instead. The first state found by DMRG is often, but not always, a single
localized Majorana. Two better starts for the second state:

- $C\psi_1$, the particle–hole conjugate (the near-zero subspace is closed under $C$), orthogonalized to $\psi_1$. It is useless when
  $\psi_1$ is itself self-conjugate (a single Majorana): then the part of $C\psi_1$ orthogonal to $\psi_1$ has norm $0.000$.
- The mirror image of $\psi_1$, $x \to N_x - 1 - x$, which maps one end of the centred chain onto the other. In the quantics encoding it is the
  bitwise complement of $i_x$: a NOT gate on every x qubit (`mirror_x`). Orthogonalized to $\psi_1$ it keeps norm $0.98$–$1.00$.

`zero_mode_subspace(…; xq)` starts the second state from the mirror image (`xq` = site numbers of the x qubits), and falls back to
$C\psi_1$ (if its orthogonal part has norm above 0.1) and then to a random start; it returns which start it used. With this start the
$2048\times2048$ case of section 8 converges to the exact values.

**Which basis of the pair DMRG returns, and its cost.** The near-zero subspace is two-dimensional, and DMRG can return any orthonormal
basis of it: two localized Majoranas, or two-ended mixtures like $\psi_f$. The two differ in cost. For the exact modes at the second
parameter point (TT-SVD with the DMRG cutoff $10^{-14}$) each Majorana needs $\chi = 121$ and $\psi_f$ needs $139$, independent of the
system size and of where the chain sits on the binary grid (shifts of 0–16 sites at $256^2$ and $512^2$). Which basis comes out is set by
the random start of $\psi_1$ and varies from size to size: in section 8, $\psi_1$ is a single Majorana at $128\times64$, $512^2$ and $2048^2$
($\chi \approx 114$; the `C_perp` column of `tn_candidate.csv` is $\approx 0$) and a mixture at $256^2$ and $1024^2$ ($\chi = 137$, $135$). Since
the DMRG cost grows as $\chi^3$, this makes the time of a size uncertain by up to $(137/114)^3 \approx 1.7$; it is why $1024^2$ takes longer
than $2048^2$ there. The mirror start of $\psi_2$ does not change it (it follows $\psi_1$), and $C\psi_1$ would steer to the two-ended
basis whenever $\psi_1$ is a mixture. Steering $\psi_1$ was tried and did not help: starting it from a random MPS projected onto the
left half of the substrate gave a Majorana at $128\times64$ and $256^2$ ($\chi = 112$, $120$), but at $512^2$, $1024^2$ and $2048^2$ the DMRG
still converged to mixtures ($\chi = 136$–$137$; `C_perp` $0.30$, $0.63$, $0.98$), and the scan took 5 % longer in total.


In [6]:
# KrylovKit warns ("ignoring imaginary component") at every DMRG step on these MPOs; the imaginary parts are round-off.
quietly(f) = with_logger(f, ConsoleLogger(stderr, Logging.Error))
norm_Hψ(H, ψ) = norm(orthogonalize(apply(H, ψ; alg="naive", truncate=false), 1)[1])
apply_site(op, ψ, j) = (φ = copy(ψ); φ[j] = noprime(op * φ[j]); φ)
τz_on(ψ) = (φ = copy(ψ); s = siteind(ψ, 1); φ[1] = noprime(ITensor([1.0 0.0; 0.0 -1.0], s', s) * φ[1]); φ)
ΔQ_mode(ψ) = real(inner(ψ, τz_on(ψ)))
mps_size(ψ) = sum(length(storage(T)) for T in ψ)              # stored elements of the MPS tensors

xqubits(g::Substrate) = (3 + g.Ly):(2 + g.Ly + g.Lx)        # site numbers of the x qubits, most significant first
"Particle–hole conjugate C·ψ = τ_x K ψ of an MPS (Nambu site first)."
Cconj(ψ) = (ns = siteind(ψ, 1); φ = dag(ψ); φ[1] = noprime(ITensor([0.0 1.0; 1.0 0.0], ns', ns) * φ[1]); φ)
"Mirror x → N_x − 1 − x: the bitwise complement of i_x, a NOT gate on each x qubit (site numbers xq)."
mirror_x(ψ, sites, xq) = (φ = copy(ψ); for j in xq; φ[j] = noprime(ITensor([0.0 1.0; 1.0 0.0], sites[j]', sites[j]) * φ[j]); end; φ)

"""
    zero_mode_subspace(HB, sites; nzero=2, η=1e-2, xq=nothing, ...) -> NamedTuple

Penalized DMRG on K = H² + η² for `nzero` near-zero states plus one gap-check state. The second state starts from the mirror
image of ψ₁ (x qubits `xq`), orthogonalized to ψ₁ — or, without `xq`, from C·ψ₁ if that has a part orthogonal to ψ₁ — and the
others from random MPS.
Returns the states, their ‖Hψ‖, the gap estimate, K, the times of building K and of the DMRG runs, and the start of state 2
(`start2` = "Cψ1", "mirror" or "random", with the norms of the orthogonalized C·ψ₁ and mirror image).
"""
function zero_mode_subspace(HB, sites; nzero=2, η=1e-2, seed=1234, nsweeps=20, maxdim=[10, 20, 50, 100, 200],
                            cutoff=1e-14, cutoff_K=1e-20, noise=[1e-6, 1e-7, 1e-8, 0.0], weight=10.0,
                            eigsolve_krylovdim=20, eigsolve_maxiter=3, xq=nothing)
    t_K = @elapsed (K = TB.build_K(HB, sites, 0.0, η; cutoff_K))
    Random.seed!(seed)
    ψs = MPS[]; start2 = ("random", NaN, NaN)
    orth(φ) = +(φ, -inner(ψs[1], φ) * ψs[1]; cutoff=1e-14)            # remove the ψ₁ component
    t_dmrg = @elapsed for k in 1:nzero+1
        ψ0 = random_mps(sites; linkdims=10)
        if k == 2
            c = orth(Cconj(ψs[1])); nc = norm(c)
            m = xq === nothing ? nothing : orth(mirror_x(ψs[1], sites, xq)); nm = m === nothing ? NaN : norm(m)
            if nm > 0.1                                  # preferred: the other end of the chain (localized Majorana)
                ψ0 = m / nm; start2 = ("mirror", nc, nm)
            elseif nc > 0.1
                ψ0 = c / nc; start2 = ("Cψ1", nc, nm)
            end
        end
        _, ψ = quietly() do
            isempty(ψs) ? dmrg(K, ψ0; nsweeps, maxdim, cutoff, noise, eigsolve_krylovdim, eigsolve_maxiter, outputlevel=0) :
                          dmrg(K, ψs, ψ0; weight, nsweeps, maxdim, cutoff, noise, eigsolve_krylovdim, eigsolve_maxiter, outputlevel=0)
        end
        push!(ψs, ψ / norm(ψ))
    end
    ψgap = pop!(ψs)
    return (; ψs, E_sub = [norm_Hψ(HB, ψ) for ψ in ψs], E_gap = norm_Hψ(HB, ψgap), K, t_K, t_dmrg, start2)
end

"""
    fermion_from_subspace(HB, sites, ψs, g; cutoff=1e-14) -> NamedTuple

X̂ localization (x qubits), Cγ = γ, and the +E fermion ψf = (γ₁ + iγ₂)/√2. Returns the Majorana x centres (0-indexed), γs, ψf, E_f.
"""
function fermion_from_subspace(HB, sites, ψs, g::Substrate; cutoff=1e-14)
    ns = sites[1]; xq = (3 + g.Ly):(2 + g.Ly + g.Lx)          # site numbers of the x qubits, most significant first
    n1(j) = ITensor([0.0 0.0; 0.0 1.0], sites[j]', sites[j])
    S = [inner(a, b) for a in ψs, b in ψs]
    X = sum(2.0^(g.Lx - k) * [inner(a, apply_site(n1(j), b, j)) for a in ψs, b in ψs] for (k, j) in enumerate(xq))
    FX = eigen(Hermitian((X + X') / 2), Hermitian((S + S') / 2))
    combine(c) = (γ = +(c[1] * ψs[1], c[2] * ψs[2]; cutoff); γ / norm(γ))
    γs = [combine(FX.vectors[:, i]) for i in 1:2]
    τx = ITensor([0.0 1.0; 1.0 0.0], ns', ns)
    C(ψ) = (φ = dag(ψ); φ[1] = noprime(τx * φ[1]); φ)
    γs = [exp(im * angle(inner(γ, C(γ))) / 2) * γ for γ in γs]   # Cγ = e^{iθ}γ → e^{iθ/2}γ is self-conjugate
    ψf = +(γs[1], im * γs[2]; cutoff); ψf = ψf / norm(ψf)
    E_f = real(inner(ψf', HB, ψf))
    E_f < 0 && ((ψf, E_f) = (C(ψf), -E_f))
    return (; centers = FX.values, γs, ψf, E_f, C)
end

fermion_from_subspace (generic function with 1 method)

## 7. Zero-mode benchmark

For every size of the width scan ($L_x = 6$, $L_y = 0\dots6$) and of the square scan ($L_x = L_y = 4\dots11$, up to $2048\times2048$
sites; $L = 6$ is the last point of the width scan), all with $\ell = 12$ and spins along $\hat z$: build $H$ and $K$, run the three
DMRGs, localize and pair. The sizes run in order of increasing $N$. Recorded: the times of each step (single thread, after a warm-up
run on a small system), the bond dimensions $\chi(K)$, $\chi(\psi)$, the memory of $K$ and of the three MPS (stored elements × 16
bytes), the peak resident memory of the Julia process after the size (`Sys.maxrss()`; it includes the Julia runtime and compiled
code, and since $N$ increases it is the peak up to and including that size), $E_f$, $\Delta Q$, the gap estimate and the Majorana
centres.

Results go to `tn_zero_modes.csv` (committed, read by the comparison notebook). With `RUN_ZM = false` the cell loads that file instead
of recomputing; progress of a run is appended to `data/zm_progress.log`. The second DMRG state starts from the mirror image of the first
(section 6); the last column of the table says which start was used.

In [7]:
RUN_ZM = true
zm_file = "tn_zero_modes.csv"
zm_cols = ["Lx", "Ly", "ell", "N", "D", "t_H", "t_K", "t_dmrg", "t_pair", "t_total", "chi_H", "chi_K", "chi_psi",
           "mem_K_MB", "mem_psi_MB", "rss_MB", "E_f", "dQ", "E_gap", "x1", "x2", "start2"]
zm_scan = sort(vcat([(6, Ly) for Ly in 0:6], [(L, L) for L in (4, 5, 7, 8, 9, 10, 11)]); by = p -> (sum(p), p[2]))   # increasing N

function zm_run(Lx, Ly; ℓ=12)
    g = Substrate(Lx, Ly, ℓ)
    t_H = @elapsed ((HB, sites) = msh_substrate(g, S_z))
    z = zero_mode_subspace(HB, sites; xq = xqubits(g))
    t_pair = @elapsed (r = fermion_from_subspace(HB, sites, z.ψs, g))
    dQ = ΔQ_mode(r.ψf)
    return Any[Lx, Ly, ℓ, nsites(g), 4nsites(g), t_H, z.t_K, z.t_dmrg, t_pair, t_H + z.t_K + z.t_dmrg + t_pair,
            maxlinkdim(HB), maxlinkdim(z.K), maximum(maxlinkdim, z.ψs), 16e-6 * mpo_size(z.K),
            16e-6 * sum(mps_size, z.ψs), Sys.maxrss() / 1e6, r.E_f, dQ, z.E_gap, r.centers..., z.start2[1]]
end

zm = if !RUN_ZM && isfile(zm_file)
    println("loading ", zm_file); readdlm(zm_file, ','; header=true)[1]
else
    zm_run(4, 0)                                                   # warm-up (compilation)
    rows = []
    open(joinpath(DATA_DIR, "zm_progress.log"), "w") do log
        for (Lx, Ly) in zm_scan
            row = zm_run(Lx, Ly); push!(rows, row)
            println(log, join(row, ", ")); flush(log)
        end
    end
    M = permutedims(reduce(hcat, rows))
    open(zm_file, "w") do io; println(io, join(zm_cols, ",")); writedlm(io, M, ','); end
    M
end

@printf("%3s %3s %8s %9s %6s %6s %7s %6s %4s %5s %5s %8s %8s %13s %13s %7s  %s\n", "Lx", "Ly", "N", "D", "t_K", "DMRG", "pair",
        "total", "χ_H", "χ_K", "χ_ψ", "K+ψ MB", "RSS MB", "E_f", "ΔQ", "gap", "start of ψ₂")
for r in eachrow(zm)
    @printf("%3d %3d %8d %9d %5.1fs %5.0fs %6.1fs %5.0fs %4d %5d %5d %8.2f %8.0f %13.6e %+13.6e %7.4f  %s\n", r[1], r[2], r[4], r[5], r[7],
            r[8], r[9], r[10], r[11], r[12], r[13], r[14] + r[15], r[16], r[17], r[18], r[19], r[22])
end

 Lx  Ly        N         D    t_K   DMRG    pair  total  χ_H   χ_K   χ_ψ   K+ψ MB   RSS MB           E_f            ΔQ     gap  start of ψ₂
  6   0       64       256   0.0s     3s    0.0s     3s    7    31     7     0.16     1240  1.491239e-03 +4.159299e-02  0.3056  mirror
  6   1      128       512   0.0s     9s    0.0s     9s    7    37    16     0.20     1489  1.868688e-04 -2.135637e-03  0.4398  mirror
  6   2      256      1024   0.0s    16s    0.0s    16s    7    38    21     0.29     1500  4.146924e-04 -3.863993e-03  0.5639  mirror
  4   4      256      1024   0.0s    15s    0.0s    16s    7    31    26     0.27     1532  3.715561e-04 -3.441196e-03  0.5529  mirror
  6   3      512      2048   0.0s    19s    0.0s    19s    7    38    21     0.32     1546  3.713820e-04 -3.448271e-03  0.5529  mirror
  6   4     1024      4096   0.0s    33s    0.0s    33s    7    38    24     0.48     1582  3.712285e-04 -3.446598e-03  0.5529  mirror
  5   5     1024      4096   0.0s    29s    0.0s  

## 8. A substrate the zero modes leak into ($\Delta = 0.6$, $\mu = -3$, $J = 3$)

At the point of section 7 the zero modes stay on the chain: $\mu = -7$ lies below the band ($\pm4.74$ with Rashba), and $\Delta = 2.4$ makes
the coherence length shorter than a site, so only $\sim2\times10^{-5}$ of the weight lies 4 or more rows away. A μ × J × Δ scan
(`msh_substrate_python.ipynb`, section 8) finds the topological region and how far the zero mode spreads; leakage needs both $\mu$ in the
band and a smaller $\Delta$. The point $\Delta = 0.6$, $\mu = -3$, $J = 3$ (with $t = 1$, $\alpha = 0.9$) is topological, gapped (gap
$0.0902$, six times smaller) and the mode spreads over several rows: 0.32 of its weight on the chain row, 0.13 four or more rows
away, decaying as $e^{-|\Delta y|/2}$. The pair splitting decays along the chain with a length of about 7 sites, so the chain is
$\ell = 64$ ($E_f/E_{\rm gap} = 10^{-4}$, comparable to $\ell = 12$ in section 7); the exact values are $E_f = 9.04771\times10^{-6}$,
$\Delta Q = 4.58344\times10^{-4}$ (converged from $128\times32$ sites on).

The same pipeline (section 6, with the mirror start) on $128\times64$ and the square substrates $256^2$ to $2048^2$. Results go to
`tn_candidate.csv` (committed; compared with MUMPS in `msh_substrate_scaling.ipynb`, section 6), one row as soon as it is done, and
sizes already in the file are skipped, so an interrupted run resumes; `RUN_CAND = false` only loads the file. About 2.5 hours.
`rss_MB` is the peak of the Julia process so far, so it includes section 7.

In [8]:
RUN_CAND = true
cand_file = "tn_candidate.csv"
const PC = (μ = -3.0, Δ = 0.6, J = 3.0)
cand_scan = [(7, 6), (8, 8), (9, 9), (10, 10), (11, 11)]
cand_cols = ["Lx", "Ly", "ell", "N", "t_H", "t_K", "t_dmrg", "t_pair", "t_total", "chi_K", "chi_psi", "mem_K_MB", "mem_psi_MB",
             "rss_MB", "E_f", "dQ", "E_gap", "E_sub1", "E_sub2", "x1", "x2", "start2", "C_perp", "mirror_perp"]

function cand_run(Lx, Ly; ℓ=64)
    g = Substrate(Lx, Ly, ℓ)
    t_H = @elapsed ((HB, sites) = msh_substrate(g, S_z; PC...))
    z = zero_mode_subspace(HB, sites; xq = xqubits(g))
    t_pair = @elapsed (r = fermion_from_subspace(HB, sites, z.ψs, g))
    return Any[Lx, Ly, ℓ, nsites(g), t_H, z.t_K, z.t_dmrg, t_pair, t_H + z.t_K + z.t_dmrg + t_pair, maxlinkdim(z.K),
               maximum(maxlinkdim, z.ψs), 16e-6 * mpo_size(z.K), 16e-6 * sum(mps_size, z.ψs), Sys.maxrss() / 1e6,
               r.E_f, ΔQ_mode(r.ψf), z.E_gap, z.E_sub..., r.centers..., z.start2...]
end

done = isfile(cand_file) ? Set((Int(r[1]), Int(r[2])) for r in eachrow(readdlm(cand_file, ','; header=true)[1])) : Set{Tuple{Int,Int}}()
if RUN_CAND
    isempty(done) && open(io -> println(io, join(cand_cols, ",")), cand_file, "w")
    for (Lx, Ly) in cand_scan
        (Lx, Ly) in done && continue
        row = cand_run(Lx, Ly)
        open(io -> println(io, join(row, ",")), cand_file, "a")
        @printf("done %d×%d: %.0f s\n", 2^Lx, 2^Ly, row[9]); flush(stdout)
    end
end

cand = readdlm(cand_file, ','; header=true)[1]
@printf("%11s %8s %7s %4s %4s %7s %13s %13s %8s %10s %10s  %s\n", "substrate", "N", "total", "χ_K", "χ_ψ", "RSS MB", "E_f", "ΔQ",
        "gap", "‖Hψ₂‖", "x₁ … x₂", "start of ψ₂ (C⊥, mirror⊥)")
for r in eachrow(cand)
    @printf("%5d×%-5d %8d %6.0fs %4d %4d %7.0f %13.6e %+13.6e %8.5f %10.2e %4.0f … %4.0f  %s (%.3f, %.3f)\n", 2^r[1], 2^r[2], r[4],
            r[9], r[10], r[11], r[14], r[15], r[16], r[17], r[19], r[20], r[21], r[22], r[23], r[24])
end

  substrate        N   total  χ_K  χ_ψ  RSS MB           E_f            ΔQ      gap      ‖Hψ₂‖    x₁ … x₂  start of ψ₂ (C⊥, mirror⊥)
  128×64        8192    516s   31  115    1814  9.047711e-06 +4.583426e-04  0.09025   9.06e-06   33 …   94  mirror (0.002, 1.000)
  512×512     262144   1536s   35  114    2089  9.047709e-06 +4.583428e-04  0.09025   9.08e-06  225 …  286  mirror (0.000, 1.000)
 2048×2048   4194304   2556s   37  114    2301  9.047709e-06 +4.583431e-04  0.09025   9.08e-06  993 … 1054  mirror (0.000, 1.000)
  256×256      65536   1340s   31  137    1941  9.047709e-06 +4.583436e-04  0.09025   9.06e-06   97 …  158  mirror (0.386, 1.000)
 1024×1024   1048576   2976s   34  135    2230  9.047709e-06 +4.583436e-04  0.09025   9.07e-06  481 …  542  mirror (0.317, 1.000)
